# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:**  Ramatou AMIDOU
**Student ID:** 30802027

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [14]:
!pip install -q openai python-dotenv pandas matplotlib

In [15]:
import os
from google.colab import userdata
from openai import OpenAI

# Get the API key securely from Colab Secrets
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

# Connect to Groq using the OpenAI-compatible API
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [16]:

def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    """
    Send a prompt to the LLM and return the complete API response.

    Parameters:
        user_prompt (str): The user's question or instruction.
        system_prompt (str): Instructions that define the assistant's role.
        temperature (float): Controls randomness in the response.
        max_tokens (int): Maximum number of tokens in the response.

    Returns:
        The complete API response.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response


# Make a simple test API call
response = ask_llm(
    "What is a loan?"
)


# Print the model's answer
print("LLM Response:")
print(response.choices[0].message.content)


# Print token usage
print("\nToken Usage:")
print(response.usage)

print("\nToken Breakdown:")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

LLM Response:
A loan is an amount of money that is borrowed from a lender, such as a bank, credit union, or individual, with the agreement to repay the principal amount, plus interest, over a specified period of time. The borrower receives the loan amount upfront and is then responsible for making regular payments, usually monthly, to pay back the loan.

Here are the key components of a loan:

1. **Principal**: The initial amount of money borrowed.
2. **Interest**: The fee charged by the lender for borrowing the money, usually expressed as a percentage of the principal amount.
3. **Repayment term**: The length of time the borrower has to repay the loan, which can range from a few months to several years.
4. **Installments**: The regular payments made by the borrower to repay the loan, which typically include both interest and principal.

Loans can be used for various purposes, such as:

* Financing a home or car purchase
* Paying for education or medical expenses
* Consolidating debt
*

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1.

The system role tells the AI how it should behave or what role it should have.
Example: “You are a helpful assistant.”

The user role contains the question or instruction given to the AI.
Example: “What is a loan?”

2.
A token is a small piece of text that the AI reads and processes. It can be a word, part of a word, or a symbol.

API providers charge by tokens because the more text the AI processes, the more computing power it uses.

### Part 1.2 — Temperature: the randomness dial

In [17]:

question = "Suggest a name for a savings product for market traders in Accra."

# Ask the question 5 times at temperature = 0.0
low_temperature_answers = []

for i in range(5):
    response = ask_llm(
        user_prompt=question,
        temperature=0.0,
        max_tokens=100
    )
    low_temperature_answers.append(response.choices[0].message.content)


# Ask the SAME question 5 times at temperature = 1.2
high_temperature_answers = []

for i in range(5):
    response = ask_llm(
        user_prompt=question,
        temperature=1.2,
        max_tokens=100
    )
    high_temperature_answers.append(response.choices[0].message.content)


# TODO: Print all 10 answers, grouped by temperature.

for i, answer in enumerate(low_temperature_answers, start=1):
    print(f"\nAnswer {i}:")
    print(answer)

for i, answer in enumerate(high_temperature_answers, start=1):
    print(f"\nAnswer {i}:")
    print(answer)


Answer 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose and target audience.
4. **

Answer 2:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect

Answer 3:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in A

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**
Temperature 0.0: The answers were very similar, often repeating the same names such as Makola Save and Trader's Treasure.

Temperature 1.2: The answers were more varied and creative, with different names such as TradeSafe, TradoSave, SokoSafe, and MakolaMmoa.

For the loan decision-support system, temperature 0.0 is more appropriate because we need consistent and reliable answers rather than creative ones.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [18]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [19]:


SUMMARY_PROMPT_V1 = "Summarize this:"

# Run V1 on L002
v1_l002 = ask_llm(
    user_prompt=f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}",
    temperature=0.7,
    max_tokens=200
)

# Run V1 on L006
v1_l006 = ask_llm(
    user_prompt=f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}",
    temperature=0.7,
    max_tokens=200
)

print("V1 — L002")
print(v1_l002.choices[0].message.content)

print("\nV1 — L006 ")
print(v1_l006.choices[0].message.content)


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_PROMPT_V2 = """
You are an assistant to a microfinance loan officer.
Summarize loan applications in a factual and neutral way.
Use only information stated in the application.
Do not invent or assume any details.
Write a short summary of 3-4 sentences.
"""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

# Run V2 on L002
v2_l002 = ask_llm(
    user_prompt=SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0,
    max_tokens=200
)

# Run V2 on L006
v2_l006 = ask_llm(
    user_prompt=SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0,
    max_tokens=200
)


# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.


print("COMPARISON: V1 vs V2")


print("\n L002 — V1 ")
print(v1_l002.choices[0].message.content)

print("\n L002 — V2 ")
print(v2_l002.choices[0].message.content)

print("\nL006 — V1 ")
print(v1_l006.choices[0].message.content)

print("\nL006 — V2 ")
print(v2_l006.choices[0].message.content)

V1 — L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when possible, despite not having collateral at the moment.

V1 — L006 
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He claims to be "business-minded" based on his friends' opinions, but has no prior experience or collateral. He promises to repay the loan in one year, once his businesses are successful, relying on his personal trustworthiness as assurance.
COMPARISON: V1 vs V2

 L002 — V1 
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**

1. V1 added details that were not clearly in the letters. For example, it said Kofi had “no experience” and wanted a “flexible repayment arrangement.” V2 stayed closer to the information given.

2. “No invented details” is important because wrong information could affect a loan decision. This problem is called hallucination

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [20]:



EXTRACT_PROMPT = """
Extract the required information from the loan application below.

Return ONLY a valid JSON object with EXACTLY these keys:

{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": true or false,
  "repayment_months": number or null
}

Rules:
- Use only information stated in the loan application.
- If a field is not stated in the letter, use null.
- Do not guess or assume information.
- amount_ghs must be a number, not a string.
- monthly_profit_ghs must be a number or null.
- has_collateral_or_guarantor must be true or false.
- repayment_months must be a number or null.
- Return ONLY the JSON object. Do not include explanations or markdown.

Worked example:

Loan application:
"My name is Ama. I run a small food shop and have operated for 5 years.
I am requesting GHS 5,000 to buy a refrigerator. I make GHS 600 profit
each month. My brother will guarantee the loan. I can repay it over 10 months."

Correct output:
{
  "applicant_name": "Ama",
  "amount_ghs": 5000,
  "purpose": "buy a refrigerator",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}

Now extract the information from this loan application:
"""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

import json

def extract_fields(letter_text):
    response = ask_llm(
        user_prompt=EXTRACT_PROMPT + letter_text,
        temperature=0,
        max_tokens=300
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if the model adds them
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    if result.endswith("```"):
        result = result[:-3]

    result = result.strip()

    try:
        return json.loads(result)

    except json.JSONDecodeError:
        print("Warning: Could not parse the model response as JSON.")
        print("Model response:")
        print(result)
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

import pandas as pd

extracted_results = {}

for letter_id, letter_text in LETTERS.items():
    extracted_results[letter_id] = extract_fields(letter_text)

extracted_df = pd.DataFrame.from_dict(
    extracted_results,
    orient="index"
)

extracted_df.index.name = "letter_id"

display(extracted_df)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1.

The example is meant to teach the model how to format the answer, not give it the answers to the actual letters. If we use one of the six letters, the model might copy information from the example instead of correctly extracting information from the new letter.

2.

Some loan letters do not contain all the required information. For example, L002 does not give a monthly profit or a specific repayment period. Without the null instruction, the model could try to guess or make up these missing values. Using null tells the model to clearly show when information is not available.

3.

Temperature 0 makes the model's answers more consistent and predictable. This is useful for extraction because we want the same information to be returned in the same format. For creative tasks, a higher temperature can be better because it allows the model to produce more varied and creative ideas.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [21]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.


BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Review the loan application and the extracted information provided below.

Prepare a decision-support brief with exactly these four sections:

1. Strengths
- List strengths supported by facts in the loan application.

2. Risks / Red Flags
- List risks or concerns supported by the application.
- Do not invent information.

3. Missing Information
- List important information or documents that the loan officer should request.

4. Suggested Next Step
- Suggest a reasonable next step, such as requesting documents,
  inviting the applicant for an interview, or flagging the application
  for senior review.
- Do NOT say "approve" or "reject".

Use only information provided in the loan application and extracted data.
Be factual and neutral.
The final loan decision must always be made by a human loan officer.
The purpose of this brief is to support the human decision, not to make
the decision.

Loan Application:
{letter_text}

Extracted Information:
{extracted_data}
"""


# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.


briefs = {}

for letter_id, letter_text in LETTERS.items():
    extracted_data = extracted_results[letter_id]

    response = ask_llm(
        user_prompt=BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_data=json.dumps(extracted_data, indent=2)
        ),
        temperature=0,
        max_tokens=500
    )

    briefs[letter_id] = response.choices[0].message.content


# Print the three required briefs


print("L001 — DECISION-SUPPORT BRIEF")

print(briefs["L001"])


print("L002 — DECISION-SUPPORT BRIEF")

print(briefs["L002"])

print("L006 — DECISION-SUPPORT BRIEF")
print(briefs["L006"])

L001 — DECISION-SUPPORT BRIEF
## 1. Strengths
- The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating stability and knowledge in her business.
- She has a steady monthly profit of GHS 900 from her current stall.
- Akosua has saved GHS 2,500 over two years with the susu scheme without missing a contribution, demonstrating her ability to save and commit to financial obligations.
- She has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.
- The applicant has a clear plan for loan repayment, proposing to pay GHS 450 monthly over 20 months.

## 2. Risks / Red Flags
- The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit and savings, which might pose a risk if her business expansion does not yield the expected increase in income.
- There is no detailed information provided about the sister's financial stability or ability to act as a guarantor, which could be a conc

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**

1. Yes. L003 has clear strengths such as an existing registered business, regular profit, sales records, and a fixed deposit. L006 has more risks because the businesses have not started, there is no collateral, and there is no profit information.

2. Practically, the AI may not have all the information needed to make a loan decision. Ethically, an incorrect decision could unfairly harm an applicant. Therefore, the final decision should remain with a human loan officer.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 252f02e


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [22]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# Fields to evaluate
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

# Store comparison results
comparison_results = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in GOLD.keys():
        gold_value = GOLD[letter_id][field]
        extracted_value = extracted_results[letter_id].get(field)

        # Compare applicant names without considering upper/lower case
        if field == "applicant_name":
            is_correct = (
                str(extracted_value).strip().lower()
                == str(gold_value).strip().lower()
            )

        # Compare numbers exactly
        elif field in ["amount_ghs", "monthly_profit_ghs", "repayment_months"]:
            if gold_value is None:
                is_correct = extracted_value is None
            else:
                is_correct = extracted_value == gold_value

        # Compare text and boolean values
        else:
            is_correct = extracted_value == gold_value

        row[letter_id] = "Correct" if is_correct else "Incorrect"

        if is_correct:
            correct_count += 1

    # Accuracy across the three gold-labelled letters
    row["accuracy"] = correct_count / len(GOLD)

    comparison_results.append(row)


# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

accuracy_df = pd.DataFrame(comparison_results)

# Convert accuracy to percentage
accuracy_df["accuracy"] = (accuracy_df["accuracy"] * 100).round(1)

display(accuracy_df)

,field,L001,L003,L006,accuracy
0,applicant_name,Correct,Correct,Correct,100.0
1,amount_ghs,Correct,Correct,Correct,100.0
2,purpose,Incorrect,Incorrect,Incorrect,0.0
3,monthly_profit_ghs,Correct,Correct,Correct,100.0
4,has_collateral_or_guarantor,Correct,Correct,Correct,100.0
5,repayment_months,Correct,Correct,Correct,100.0


### Part 4.2 — Reliability: is the system consistent?

In [23]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

import json

# Run L004 five times at temperature = 0
results_temp_0 = []

for i in range(5):
    result = extract_fields(LETTERS["L004"])
    results_temp_0.append(result)


# Run L004 five times at temperature = 1.0
results_temp_1 = []

for i in range(5):
    # We need to call the LLM directly here so we can change the temperature
    response = ask_llm(
        user_prompt=EXTRACT_PROMPT + LETTERS["L004"],
        temperature=1.0,
        max_tokens=300
    )

    result = response.choices[0].message.content.strip()

    # Remove JSON markdown fences if present
    if result.startswith("```json"):
        result = result[7:]
    elif result.startswith("```"):
        result = result[3:]

    if result.endswith("```"):
        result = result[:-3]

    try:
        result = json.loads(result.strip())
    except json.JSONDecodeError:
        print(f"Warning: Temperature 1.0, run {i+1} produced invalid JSON.")
        result = None

    results_temp_1.append(result)


# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.


def reliability_summary(results):
    # Count valid JSON results
    valid_results = [r for r in results if isinstance(r, dict)]

    valid_json_count = len(valid_results)

    # Convert valid results to sorted JSON strings
    unique_results = set(
        json.dumps(r, sort_keys=True)
        for r in valid_results
    )

    # Number of runs with the most common identical result
    if valid_results:
        counts = {}

        for r in valid_results:
            result_string = json.dumps(r, sort_keys=True)
            counts[result_string] = counts.get(result_string, 0) + 1

        identical_count = max(counts.values())
    else:
        identical_count = 0

    return valid_json_count, identical_count, len(unique_results)


# Calculate reliability for both temperatures
temp_0_valid, temp_0_identical, temp_0_unique = reliability_summary(
    results_temp_0
)

temp_1_valid, temp_1_identical, temp_1_unique = reliability_summary(
    results_temp_1
)


# Display results
print("Reliability Results ")

print("\nTemperature = 0.0")
print(f"Valid JSON: {temp_0_valid}/5")
print(f"Identical values across runs: {temp_0_identical}/5")
print(f"Unique outputs: {temp_0_unique}")

print("\nTemperature = 1.0")
print(f"Valid JSON: {temp_1_valid}/5")
print(f"Identical values across runs: {temp_1_identical}/5")
print(f"Unique outputs: {temp_1_unique}")

Reliability Results 

Temperature = 0.0
Valid JSON: 5/5
Identical values across runs: 5/5
Unique outputs: 1

Temperature = 1.0
Valid JSON: 5/5
Identical values across runs: 5/5
Unique outputs: 1


### Part 4.3 — Hallucination probing

In [25]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?


# Test 1 — Ask about information that is NOT in the letter


test1_question = """
What is the applicant's credit score?
If the credit score is not stated in the letter, say that it is not provided.
"""

test1_response = ask_llm(
    user_prompt=f"{test1_question}\n\nLoan application:\n{LETTERS['L002']}",
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0,
    max_tokens=100
)

test1_output = test1_response.choices[0].message.content

print("TEST 1 — Missing Information")

print("Question:")
print(test1_question)
print("\nOutput:")
print(test1_output)


# Test 2 — Give the extractor irrelevant information

irrelevant_text = """
The weather forecast for Accra today is partly cloudy with a chance
of rain in the afternoon. Temperatures are expected to remain warm.
Residents are advised to carry umbrellas.
"""

test2_response = ask_llm(
    user_prompt=EXTRACT_PROMPT + irrelevant_text,
    temperature=0,
    max_tokens=300
)

test2_output = test2_response.choices[0].message.content.strip()

# Remove markdown JSON fences if present
if test2_output.startswith("```json"):
    test2_output = test2_output[7:]
elif test2_output.startswith("```"):
    test2_output = test2_output[3:]

if test2_output.endswith("```"):
    test2_output = test2_output[:-3]

test2_output = test2_output.strip()


print("TEST 2 — Irrelevant Input")
print("Input:")
print(irrelevant_text)
print("\nOutput:")
print(test2_output)


# TODO: Record the outputs verbatim below and label each PASS or FAIL.


print("HALLUCINATION TEST RESULTS")

print("Test 1: PASS - The model correctly stated that the credit score was not provided and did not invent one.")
print("Test 2: PASS -The extractor returned null for the missing loan information and did not fabricate an applicant or loan details.")

TEST 1 — Missing Information
Question:

What is the applicant's credit score?
If the credit score is not stated in the letter, say that it is not provided.


Output:
Kwame Boateng, a commercial driver, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season. The applicant does not have collateral to offer at the moment. His credit score is not provided in the application.
TEST 2 — Irrelevant Input
Input:

The weather forecast for Accra today is partly cloudy with a chance
of rain in the afternoon. Temperatures are expected to remain warm.
Residents are advised to carry umbrellas.


Output:
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": null
}
HALLUCINATION TEST RESULTS
Test 1: PASS - The model correctly stated that the credit score was not provided and did not invent one.
Test 2:

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**


1. The extraction accuracy was 83.3% overall. The hardest field was purpose because the model used different wording from the gold answers, even though the meaning was the same.

2. At both temperatures, all 5/5 runs produced valid JSON and identical results. This shows the system was consistent in this test. However, production systems should still be tested regularly because consistency does not always mean correctness.

3. No, the system did not hallucinate in our tests. It correctly said the credit score was not provided and returned null for missing information. Using instructions like “do not guess”, requiring null for missing fields, and keeping a human in the loop can reduce hallucination risk.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**

1. Applicants who write poor English could be unfairly harmed if the system mistakes poor writing for a weak application. A good business owner could therefore be rejected even though their business is actually strong.

2. Sending loan letters to a foreign API could expose applicants' personal and financial information to a third party. Before deployment, I would check the API provider's privacy and security practices, data storage location, data retention, and whether the provider can use our data for training. I would also check compliance with Ghana's Data Protection Act, 2012 (Act 843) and the requirements of the Data Protection Commission. Ghana's DPC requires organisations processing personal data to register and use appropriate security and organisational safeguards.

3. Two safeguards I would use are:

Human review: A loan officer must review the AI's output before any final decision.

Monitoring and appeals: Keep logs of AI outputs and allow applicants to challenge or request a human review of decisions.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**


1.

Prompting is similar to tuning model hyperparameters because we test different settings and compare the results. The difference is that prompting changes the instructions given to the model, while hyperparameters change how a model learns or generates outputs.

2.

I would not trust the system to run unattended. Although it was consistent and did not hallucinate in our tests, the extraction accuracy was not perfect, especially for the purpose field. A human should therefore remain involved in the final decision.

3.

My first API call used 384 tokens. At 1,000 applications, this would be roughly 384,000 tokens per month if each application required a similar amount of usage. This suggests that a provider with a generous free tier or low token cost would be important when choosing an API.

4.

Calling an API is better for this task because a powerful model is already trained, so we can use it without collecting a large dataset, training a model, or spending significant computing resources. Training our own model would make more sense if we had a large specialized dataset, needed full control over the model, or had privacy requirements that made using a third-party API unsuitable.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.